<a href="https://colab.research.google.com/github/Munya574/Wealth_of_Healh_and_Nutrition/blob/phase1-data-cleaning/notebooks/K_MeansClustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np

In [7]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Load the heatlh dataset
health_dataset = pd.read_csv("nhanes_clean.csv")

def cluster_health_patterns(df, n_clusters, feature_columns):
    """Run k-Means on `feature_columns` of `df`. Standardizes features first
    (so a 30000 MW load doesn't dominate a 1000 MW solar feature). Returns
    the dataframe with a new `energy_cluster` column."""
    df = df.copy()
    df_clean = df[feature_columns].fillna(df[feature_columns].mean())
    scaled = StandardScaler().fit_transform(df_clean)
    model = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    df["health_cluster"] = model.fit_predict(scaled)
    return df

# Show what's available
numeric_columns = list(health_dataset.select_dtypes(include="number").columns)
print(f"health_dataset: {health_dataset.shape[0]} rows x {health_dataset.shape[1]} columns")
print(f"Numeric columns you could cluster on: {numeric_columns}")

health_dataset: 25225 rows x 36 columns
Numeric columns you could cluster on: ['SEQN', 'sex', 'age', 'race_eth', 'education', 'income_poverty_ratio', 'bmi', 'waist_cm', 'bp_systolic', 'bp_diastolic', 'hba1c', 'hdl', 'total_chol', 'triglycerides', 'ldl', 'energy_kcal', 'protein_g', 'carb_g', 'sugar_g', 'fiber_g', 'fat_total_g', 'fat_sat_g', 'sodium_mg', 'vigorous_rec_activity', 'moderate_rec_activity', 'sedentary_min', 'smoked_100_cigs', 'avg_drinks_per_day', 'food_security_adult', 'weight_mec_2yr', 'weight_int_2yr', 'svy_psu', 'svy_stratum', 'sleep_hours']


In [8]:
# TODO: Integer number of clusters. (Try 3 first, then 5, then 2.)
my_k = 5

# TODO: List of numeric column names to cluster on.
my_features = ["age", "sex", "race_eth", "education", "income_poverty_ratio",
    "food_security_adult"]

# Run the clustering.
clustered_df = cluster_health_patterns(health_dataset, n_clusters=my_k, feature_columns=my_features)

# Inspect what k-Means came up with:
#   - How many rows ended up in each cluster?
#   - What do the cluster centers look like in real units?
print(f"Cluster sizes (k={my_k}):")
print(clustered_df["health_cluster"].value_counts().sort_index())

print(f"\nMean of each feature per cluster:")
print(clustered_df.groupby("health_cluster")[my_features].mean().round(1))

Cluster sizes (k=5):
health_cluster
0    4565
1    5856
2    3838
3    5687
4    5279
Name: count, dtype: int64

Mean of each feature per cluster:
                 age  sex  race_eth  education  income_poverty_ratio  \
health_cluster                                                         
0               23.4  1.5       3.0        3.5                   1.2   
1               16.8  2.0       3.4        3.8                   2.2   
2               59.4  1.5       2.7        1.9                   1.5   
3               15.4  1.0       3.4        3.6                   2.3   
4               55.7  1.5       3.8        4.5                   4.0   

                food_security_adult  
health_cluster                       
0                               3.5  
1                               1.2  
2                               1.7  
3                               1.2  
4                               1.1  
